# Niche hypotheses and conditional fate responses

**Role:** Analysis.

Read previously exported result tables describing recipient specificity, precursor/mature fate contrasts and Lysozyme/Serotonin redundancy while preserving recipient identity. Interpret these as model associations, alongside measured crypt context and support across size/timepoint. Optional FiLM-only diagnostics separate conditioning from head effects and assess factorial contrasts. They apply specifically to the FiLM architecture; the general embedding analysis remains in embeddings/. No training is repeated.

This notebook displays archived study reports only; it does not load models or run ablations. For new inference with saved training runs, use `../ablation/total_analysis.ipynb` or `../ablation/size_dependent_ablation.ipynb`.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.


## Analysis settings

Choose the input run or cached report and all analysis options here before model loading, inference or plotting. Derived paths and model-dependent defaults are resolved in the following cells; each checkpoint restores its saved data and transformations.

In [ ]:
import numpy as np
import torch

# Execution and display
SHOW_FILM_ROUTES = False  # Display the optional FiLM/head-route comparison.

# Input runs, data and destinations
LGR5_RUN_NAME = '20260910_164825_all_depth2_four_models_baseline_with_size'  # Name of the new run or the explicitly selected saved run.
NICHE_RUN_DIR = PROJECT_ROOT / "results_experiments/size_conditioned_ablation/20260910_164825_all_depth2_four_models_baseline_with_size"  # Training run to create or saved run to analyze.
NICHE_OUTPUT_DIR = NICHE_RUN_DIR / "niche_hypotheses"  # Directory for this analysis’s tables and figures.

# Inference resources
LGR5_DEVICE = 'cuda'  # Inference/training device; CUDA must be available for cuda.
LGR5_BATCH_SIZE = 256  # Number of graphs or subgraphs per batch.

# Statistical support
LGR5_MIN_ORGANOIDS = 10  # Minimum distinct organoids required for reported effects.
LGR5_BOOTSTRAP_SAMPLES = 1000  # Number of organoid bootstrap resamples.


## Niche figure definitions

These study-specific definitions build the summaries or panels displayed below. Cohort selection, marker labels, thresholds and figure layouts remain visible here; shared numerical and plotting primitives are imported from `src/`. Defining these functions does not run inference or load results.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter

NICHE_COLORS = {
    'Agr2': '#d99a22',  # Display color for Agr2.
    'Lysozyme': '#2865ad',  # Display color for Lysozyme.
    'Chroma': '#9b51a0',  # Display color for Chroma.
    'Serotonin': '#c84c4c',  # Display color for Serotonin.
    'precursor_only': '#d99a22',  # Display color for precursor_only.
    'double_positive': '#2865ad',  # Display color for double_positive.
    'mature_only': '#c84c4c',  # Display color for mature_only.
}  # Display colors assigned to groups or model variants.

def niche_curve(ax,data,label,color=None,style='-'):
    data=data[data.n>0].sort_values('n')
    if data.empty:return
    ax.plot(data.n,data['mean'],style,label=label,color=color,lw=2)
    ax.fill_between(data.n,data.ci_low,data.ci_high,color=color,alpha=.13)

def niche_finish(ax,ylabel='Normalized curvature change',xlabel='Supplied cell count N'):
    ax.set_xscale('log'); ax.set_xticks([165,244,361,535,791],[165,244,361,535,791])
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.axhline(0,color='.5',lw=.7); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.grid(alpha=.15); ax.legend(fontsize=8,frameon=False)

def niche_build_figures(out):
    out=Path(out); figs={}
    def read(name):return pd.read_csv(out/f'{name}_summary.csv')
    unit='delta_relative'
    d=read('specificity');d=d[d.source_state=='all_positive']
    fig,axes=plt.subplots(2,2,figsize=(12,8),layout='constrained')
    for i,marker in enumerate(['Lysozyme','Serotonin']):
        for j,hop in enumerate([1,2]):
            ax=axes[i,j];g=d[(d.marker==marker)&(d.hop==hop)]
            for metric,label,color in [('lgr5_'+unit,'LGR5+ recipient','#2865ad'),('other_'+unit,'LGR5-negative recipient','#c84c4c')]:
                niche_curve(ax,g[g.metric==metric],label,color)
            niche_finish(ax);ax.set_title(f'{marker} source → hop {hop}; paired source cohort')
    fig.suptitle('Edit one source marker; compare unchanged recipients around the same source')
    figs['01_recipient_fields']=fig
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,hop in zip(axes,[1,2]):
        for marker in ['Lysozyme','Serotonin']:
            niche_curve(ax,d[(d.marker==marker)&(d.hop==hop)&(d.metric==unit)],marker,NICHE_COLORS[marker])
        niche_finish(ax,'Effect at LGR5+ minus effect at LGR5-negative');ax.set_title(f'Paired recipient specificity, hop {hop}')
    figs['02_recipient_specificity']=fig
    d=read('lineage')
    fig,axes=plt.subplots(2,2,figsize=(12,8),layout='constrained')
    for row,(a,b) in enumerate([('Agr2','Lysozyme'),('Chroma','Serotonin')]):
        for col,hop in enumerate([1,2]):
            ax=axes[row,col]
            for marker,state,label,style in [(a,'precursor_only',f'{a}+ / {b}−: edit {a}','-'),
                (a,'double_positive',f'double+: edit {a}','--'),(b,'double_positive',f'double+: edit {b}','-'),
                (b,'mature_only',f'{a}− / {b}+: edit {b}',':')]:
                niche_curve(ax,d[(d.marker==marker)&(d.source_state==state)&(d.hop==hop)&(d.metric==unit)],label,NICHE_COLORS[marker],style)
            niche_finish(ax);ax.set_title(f'{a} / {b} source states → LGR5+, hop {hop}')
    fig.suptitle('Separate marker states; these curves have different eligible cohorts')
    figs['03_source_states']=fig
    d=read('dominance')
    fig,axes=plt.subplots(2,3,figsize=(16,8),layout='constrained')
    pairs=[('Agr2','Lysozyme'),('Chroma','Serotonin'),('Lysozyme','Serotonin')]
    for col,(a,b) in enumerate(pairs):
        for row,hop in enumerate([1,2]):
            ax=axes[row,col];g=d[(d.marker_a==a)&(d.marker_b==b)&(d.hop==hop)]
            for metric,label,color in [('dominance_delta_relative','Geometrically normalized','#2865ad'),('dominance_delta_z','Transformed target','#c84c4c')]:
                niche_curve(ax,g[g.metric==metric],label,color)
            niche_finish(ax,f'D: positive favors {a}',xlabel='Supplied cell count N')
            ax.set_ylim(-1,1);support=g.n_organoids.max()
            ax.set_title(f'{a} vs {b}, hop {hop}; {support} organoids')
    fig.suptitle('Matched recipients: relative strength D = (mean |effect A| − mean |effect B|) / their sum')
    figs['04_matched_dominance']=fig
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,(a,b) in zip(axes,pairs[:2]):
        g=d[(d.marker_a==a)&(d.marker_b==b)&(d.hop==1)]
        for marker,prefix in [(a,'a_'),(b,'b_')]:
            niche_curve(ax,g[g.metric==prefix+unit],marker,NICHE_COLORS[marker])
        niche_finish(ax);ax.set_title(f'{a} vs {b}: signed effects on the same LGR5 recipients')
    figs['05_matched_signed_effects']=fig
    d=read('backup');p=read('pairs')
    fig,axes=plt.subplots(2,2,figsize=(12,8),layout='constrained')
    for i,marker in enumerate(['Lysozyme','Serotonin']):
        ax=axes[0,i];g=d[(d.marker==marker)&(d.hop==1)]
        for prefix,label,color in [('backup0_','No other backup within 2 hops','#c84c4c'),('backup1_','Other backup within 2 hops','#2865ad')]:
            niche_curve(ax,g[g.metric==prefix+unit],label,color)
        niche_finish(ax);ax.set_title(f'Edit {marker}; matched context strata, hop 1')
    ax=axes[1,0];g=p[(p.selection=='all')&(p.hop==1)]
    for metric,label,color in [('lyso_relative','Remove Lysozyme','#2865ad'),('sero_relative','Remove Serotonin','#c84c4c'),('joint_relative','Remove both','#333333')]:
        niche_curve(ax,g[g.metric==metric],label,color)
    niche_finish(ax);ax.set_title('Distinct-source factorial experiment; unchanged LGR5 recipient')
    ax=axes[1,1]
    for selection,label,color in [('all','All selected pairs','#2865ad'),('one_each_in_hop','Exactly one of each in the tested hop','#c84c4c')]:
        niche_curve(ax,p[(p.selection==selection)&(p.hop==1)&(p.metric=='interaction_relative')],label,color)
    niche_finish(ax,'Joint − Lysozyme-only − Serotonin-only effect');ax.set_title('Nonadditivity of dual marker removal')
    figs['06_redundancy']=fig
    d=read('crypt');d=d[(d.source_state=='all_positive')&(d.metric==unit)]
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,marker in zip(axes,['Lysozyme','Serotonin']):
        for region,label,color in [('detected_crypt','In detected crypt','#2865ad'),('outside_detected_crypt','Outside detected crypt','#d99a22'),('no_crypt_detected','No crypt detected in organoid','#c84c4c')]:
            niche_curve(ax,d[(d.marker==marker)&(d.hop==1)&(d.recipient_region==region)],label,color)
        niche_finish(ax);ax.set_title(f'{marker} source → LGR5+ recipient, hop 1')
    fig.suptitle('Observed crypt labels held fixed throughout each hypothetical size sweep')
    figs['07_crypt_context']=fig
    d=pd.read_csv(out/'dominance_observed_n_bin.csv');orgs=pd.read_csv(out/'organoid_metadata.csv')
    centers=orgs.groupby('n_bin').observed_n.median()
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,(a,b) in zip(axes,pairs[:2]):
        g=d[(d.marker_a==a)&(d.marker_b==b)&(d.hop==1)]
        for metric,label,color in [('dominance_delta_relative','Geometrically normalized','#2865ad'),('dominance_delta_z','Transformed target','#c84c4c')]:
            q=g[g.metric==metric].sort_values('n_bin');xx=q.n_bin.map(centers)
            ax.errorbar(xx,q['mean'],yerr=[q['mean']-q.ci_low,q.ci_high-q['mean']],fmt='o-',label=label,color=color,capsize=3)
        ax.axhline(0,color='.5',lw=.7);ax.set_xlabel('Median observed N in cohort quartile');ax.set_ylabel(f'D: positive favors {a}')
        ax.set_title(f'{a} vs {b}: observed-size predictions');ax.legend(fontsize=8);ax.grid(alpha=.15)
    fig.suptitle('Cross-sectional comparison: different organoids occur in different size bins')
    figs['08_observed_dominance']=fig
    d=read('prevalence');days=['day3p5','day4','day4p5','day4p5-more']
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,family in zip(axes,['Lysozyme','Serotonin']):
        for state in ['precursor_only','double_positive','mature_only']:
            q=d[(d.family==family)&(d.source_state==state)&(d.region=='detected_crypt')].set_index('timepoint').reindex(days)
            ax.errorbar(range(4),q['mean'],yerr=[q['mean']-q.ci_low,q.ci_high-q['mean']],fmt='o-',label=state.replace('_',' '),color=NICHE_COLORS[state],capsize=3)
        ax.set_xticks(range(4),days,rotation=15);ax.set_ylabel('Cell fraction within detected crypt regions')
        ax.set_title(f'{family} family: measured marker states');ax.legend(fontsize=8);ax.grid(alpha=.15)
    fig.suptitle('Destructive snapshots; day4p5 and day4p5-more are kept as separate acquisition groups')
    figs['09_state_prevalence']=fig
    d=read('measured_curvature')
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,region in zip(axes,['detected_crypt','outside_detected_crypt']):
        for marker in ['LGR5','Lysozyme','Serotonin']:
            q=d[(d.marker==marker)&(d.region==region)].set_index('timepoint').reindex(days)
            ax.errorbar(range(4),q['mean'],yerr=[q['mean']-q.ci_low,q.ci_high-q['mean']],fmt='o-',label=marker,capsize=3)
        ax.set_xticks(range(4),days,rotation=15);ax.set_ylabel('Measured Gaussian curvature (dataset units)');ax.set_title(region.replace('_',' '));ax.legend();ax.grid(alpha=.15)
    fig.suptitle('Measured cell-patch curvature; marker groups overlap and do not measure apical constriction')
    figs['10_measured_curvature']=fig
    d=pd.read_csv(out/'dominance_local_slope_summary.csv')
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,hop in zip(axes,[1,2]):
        q=d[(d.hop==hop)&(d.metric=='dominance_delta_z')].copy()
        labels=q.marker_a+' vs '+q.marker_b
        ax.errorbar(q['mean'],range(len(q)),xerr=[q['mean']-q.ci_low,q.ci_high-q['mean']],fmt='o',capsize=3)
        ax.set_yticks(range(len(q)),labels);ax.axvline(0,color='.5',lw=.7)
        ax.set_xlabel('Local change in D per unit log N');ax.set_title(f'Bracketing actual N, hop {hop}');ax.grid(alpha=.15)
    fig.suptitle('Local sweep check: only graphs with observed N inside the supplied grid')
    figs['11_local_size_sensitivity']=fig
    d=pd.read_csv(out/'dominance_observed_timepoint.csv')
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,(a,b) in zip(axes,pairs[:2]):
        for metric,label,color in [('dominance_delta_relative','Geometrically normalized','#2865ad'),('dominance_delta_z','Transformed target','#c84c4c')]:
            q=d[(d.marker_a==a)&(d.marker_b==b)&(d.hop==1)&(d.metric==metric)].set_index('timepoint').reindex(days)
            ax.errorbar(range(4),q['mean'],yerr=[q['mean']-q.ci_low,q.ci_high-q['mean']],fmt='o-',label=label,color=color,capsize=3)
        ax.set_xticks(range(4),days,rotation=15);ax.axhline(0,color='.5',lw=.7)
        ax.set_ylabel(f'D: positive favors {a}');ax.set_title(f'{a} vs {b}, observed size');ax.legend(fontsize=8);ax.grid(alpha=.15)
    fig.suptitle('Snapshot acquisition groups; these comparisons are not longitudinal or size-matched')
    figs['12_snapshot_timepoints']=fig
    d=read('crypt_paired')
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,hop in zip(axes,[1,2]):
        for marker in ['Lysozyme','Serotonin']:
            niche_curve(ax,d[(d.marker==marker)&(d.hop==hop)&(d.metric==unit)],marker,NICHE_COLORS[marker])
        niche_finish(ax,'Effect inside minus outside detected crypt');ax.set_title(f'Same source; LGR5 recipients in both regions, hop {hop}')
    figs['13_paired_crypt_context']=fig
    d=read('recipient_types')
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,marker in zip(axes,['Lysozyme','Serotonin']):
        for phenotype,color in zip(['AldoB','KI67','Agr2','Chroma','Lysozyme','Serotonin'],plt.cm.tab10.colors):
            niche_curve(ax,d[(d.marker==marker)&(d.hop==1)&(d.phenotype==phenotype)&(d.metric==unit)],phenotype,color)
        niche_finish(ax);ax.set_title(f'{marker} source → LGR5-negative recipient phenotypes')
    fig.suptitle('Descriptive phenotype subsets overlap and have different source cohorts')
    figs['14_other_recipient_phenotypes']=fig
    p=read('pairs');b=read('backup')
    fig,axes=plt.subplots(2,2,figsize=(12,8),layout='constrained')
    for col,hop in enumerate([1,2]):
        for marker in ['Lysozyme','Serotonin']:
            niche_curve(axes[0,col],b[(b.marker==marker)&(b.hop==hop)&(b.metric=='delta_z')],marker,NICHE_COLORS[marker])
        niche_finish(axes[0,col],'Backup present − absent effect (transformed)');axes[0,col].set_title(f'Matched backup context, hop {hop}')
        for selection,label,color in [('all','All selected pairs','#2865ad'),('one_each_in_hop','Exactly one of each in tested hop','#c84c4c')]:
            niche_curve(axes[1,col],p[(p.selection==selection)&(p.hop==hop)&(p.metric=='interaction_z')],label,color)
        niche_finish(axes[1,col],'Factorial interaction (transformed)');axes[1,col].set_title(f'Distinct-source nonadditivity, hop {hop}')
    figs['15_transformed_redundancy_checks']=fig
    return figs

def niche_save_figures(out):
    out=Path(out); directory=out/'figures';directory.mkdir(exist_ok=True)
    items=niche_build_figures(out)
    for name,fig in items.items():
        fig.savefig(directory/f'{name}.png',dpi=160,bbox_inches='tight')
        fig.savefig(directory/f'{name}.pdf',bbox_inches='tight')
    return items

## Niche study summaries

These study-specific definitions build the summaries or panels displayed below. Cohort selection, marker labels, thresholds and figure layouts remain visible here; shared numerical and plotting primitives are imported from `src/`. Defining these functions does not run inference or load results.

In [ ]:
NICHE_STATISTICS_METRICS = ['delta_z','delta_raw','delta_relative']  # Quantities included in the study summaries.

import itertools
import json
from src.analysis.conditioning.response_statistics import cluster_summary


def niche_statistics_summarize(out, draws=1000):
    out=Path(out)
    sm=pd.read_csv(out/'recipient_manifest.csv.gz')
    pm=pd.read_csv(out/'pair_manifest.csv.gz')
    cells=pd.read_csv(out/'cell_census.csv.gz')
    config=json.loads((out/'settings.json').read_text()); counts=config['counts']
    # Sampling and descriptive prevalence refer to actual observed sizes only.
    orgs=cells[['organoid_str','observed_n','timepoint','dataset']].drop_duplicates()
    edges=np.unique(np.quantile(orgs.observed_n,[0,.25,.5,.75,1])); edges[0]-=1; edges[-1]+=1
    orgs['n_bin']=pd.cut(orgs.observed_n,edges,labels=False)
    nb=orgs.set_index('organoid_str').n_bin
    cells['n_bin']=cells.organoid_str.map(nb)
    sm['n_bin']=sm.organoid_str.map(nb)
    sm['recipient_class']=np.where(sm.recipient_lgr5==1,'LGR5+','LGR5−')
    sm['degree_bin']=(sm.recipient_degree//3).astype(int)
    sm['lgr5_bin']=np.minimum((sm.local_lgr5_fraction*4).astype(int),3)
    # Exact coarsened context strata. Same organoid controls N, timepoint and batch.
    context_keys=['organoid_str','source','hop','source_state','recipient_markers','recipient_region','degree_bin','lgr5_bin']
    context_meta=[]
    for marker, backup in [('Lysozyme','sero'),('Serotonin','lyso')]:
        m=sm[(sm.marker==marker)&(sm.recipient_lgr5==1)&(sm[f'source_{backup}']==0)].copy()
        m['backup_present']=(m[f'backup_{backup}']>0).astype(int)
        eligible=m.groupby(context_keys).backup_present.nunique()
        eligible=eligible[eligible==2].reset_index()[context_keys]
        m=m.merge(eligible,on=context_keys,validate='many_to_one')
        context_meta.append(m[['case_id','backup_present']])
    cm=pd.concat(context_meta,ignore_index=True)
    cm.to_csv(out/'backup_matched_cases.csv',index=False)
    metadata=dict(n_bin_edges=edges.tolist(),bootstrap_draws=draws,
        bootstrap_unit='organoid; fitted seeds averaged, not independent replicates',
        backup_matching=context_keys,backup_excludes_source_coexpressing_backup=True)
    (out/'summary_settings.json').write_text(json.dumps(metadata,indent=2))
    orgs.to_csv(out/'organoid_metadata.csv',index=False)
    # Cell fractions, measured curvature and marker-state availability: equal organoid weighting.
    census=[]
    for family,col in [('Lysozyme','lyso_state'),('Serotonin','sero_state')]:
        for state in ['precursor_only','double_positive','mature_only']:
            temp=cells.assign(fraction=(cells[col]==state).astype(float))
            g=temp.groupby(['organoid_str','timepoint','n_bin','region'],observed=True).agg(
                fraction=('fraction','mean'),n_cells=('fraction','size')).reset_index()
            census.append(g.assign(family=family,source_state=state))
    census=pd.concat(census,ignore_index=True)
    census.to_csv(out/'census_by_organoid.csv',index=False)
    cluster_summary(census,['timepoint','region','family','source_state'],['fraction'],draws=draws).to_csv(out/'prevalence_summary.csv',index=False)
    measured=[]
    for marker in ['LGR5','Agr2','Lysozyme','Chroma','Serotonin']:
        g=cells[cells[marker]==1].groupby(['organoid_str','timepoint','n_bin','region']).curvature.mean().reset_index()
        measured.append(g.assign(marker=marker))
    measured=pd.concat(measured,ignore_index=True)
    measured.to_csv(out/'measured_curvature_by_organoid.csv',index=False)
    cluster_summary(measured,['timepoint','region','marker'],['curvature'],draws=draws).to_csv(out/'measured_curvature_summary.csv',index=False)
    results={k:[] for k in ['field','specificity','lineage','dominance','backup','pairs','crypt','crypt_paired','timepoint','observed','recipient_types']}
    dominance_cases=[]
    support=[]
    def store(name, frame, keys, metrics):
        if len(frame):
            grouped=frame.groupby(['organoid_str',*keys],observed=True,dropna=False)[metrics].mean().reset_index()
            results[name].append(grouped)
    for fold in sorted(sm.fold.unique()):
        ms=sm[sm.fold==fold]; mp=pm[pm.fold==fold]
        for seed in config['seeds']:
            for n in counts+['observed']:
                suffix=f'fold_{fold}_seed_{seed}_N{n}'
                s=pd.read_csv(out/f'{suffix}_singles.csv.gz').merge(ms,on='case_id',validate='one_to_one')
                p=pd.read_csv(out/f'{suffix}_pairs.csv.gz').merge(mp,on='case_id',validate='one_to_one')
                assert len(s)==len(ms) and len(p)==len(mp)
                s['seed']=seed; s['n']=0 if n=='observed' else n
                p['seed']=seed; p['n']=s.n.iloc[0]
                basekeys=['seed','n']
                # Average recipients within each edited source first, then source cells within organoid.
                sourcekeys=['organoid_str','source','marker','source_state','hop','recipient_class',*basekeys]
                sf=s.groupby(sourcekeys,observed=True)[NICHE_STATISTICS_METRICS].mean().reset_index()
                store('field',sf,['marker','source_state','hop','recipient_class',*basekeys],NICHE_STATISTICS_METRICS)
                store('field',sf.assign(source_state='all_positive'),['marker','source_state','hop','recipient_class',*basekeys],NICHE_STATISTICS_METRICS)
                if n=='observed':
                    store('observed',s[s.recipient_lgr5==1],['marker','source_state','hop','n_bin',*basekeys],NICHE_STATISTICS_METRICS)
                    store('timepoint',s[s.recipient_lgr5==1],['marker','source_state','hop','timepoint',*basekeys],NICHE_STATISTICS_METRICS)
                    store('observed',s[s.recipient_lgr5==1].assign(source_state='all_positive'),['marker','source_state','hop','n_bin',*basekeys],NICHE_STATISTICS_METRICS)
                    store('timepoint',s[s.recipient_lgr5==1].assign(source_state='all_positive'),['marker','source_state','hop','timepoint',*basekeys],NICHE_STATISTICS_METRICS)
                # Paired LGR5+ minus LGR5-negative recipients around exactly the same source/hop.
                ix=['organoid_str','source','marker','source_state','hop',*basekeys]
                wide=sf.pivot(index=ix,columns='recipient_class',values=NICHE_STATISTICS_METRICS).dropna()
                contrast=pd.DataFrame({m:wide[(m,'LGR5+')]-wide[(m,'LGR5−')] for m in NICHE_STATISTICS_METRICS}).reset_index()
                for m in NICHE_STATISTICS_METRICS:
                    contrast['lgr5_'+m]=wide[(m,'LGR5+')].to_numpy()
                    contrast['other_'+m]=wide[(m,'LGR5−')].to_numpy()
                specificity_metrics=NICHE_STATISTICS_METRICS+['lgr5_'+m for m in NICHE_STATISTICS_METRICS]+['other_'+m for m in NICHE_STATISTICS_METRICS]
                store('specificity',contrast,['marker','source_state','hop',*basekeys],specificity_metrics)
                store('specificity',contrast.assign(source_state='all_positive'),['marker','source_state','hop',*basekeys],specificity_metrics)
                # Secondary recipient phenotypes overlap; LGR5+ recipients are excluded from all comparators.
                for phenotype in ['AldoB','KI67','Agr2','Chroma','Lysozyme','Serotonin']:
                    mask=(s.recipient_lgr5==0)&s.recipient_markers.fillna('').str.split('|').apply(lambda a: phenotype in a)
                    t=s[mask].groupby(['organoid_str','source','marker','hop',*basekeys])[NICHE_STATISTICS_METRICS].mean().reset_index()
                    store('recipient_types',t.assign(phenotype=phenotype),['marker','hop','phenotype',*basekeys],NICHE_STATISTICS_METRICS)
                l=s[s.recipient_lgr5==1].copy()
                store('lineage',l,['marker','source_state','hop',*basekeys],NICHE_STATISTICS_METRICS)
                store('crypt',l,['marker','source_state','hop','recipient_region',*basekeys],NICHE_STATISTICS_METRICS)
                store('lineage',l.assign(source_state='all_positive'),['marker','source_state','hop',*basekeys],NICHE_STATISTICS_METRICS)
                store('crypt',l.assign(source_state='all_positive'),['marker','source_state','hop','recipient_region',*basekeys],NICHE_STATISTICS_METRICS)
                cr=l.groupby(['organoid_str','source','marker','hop',*basekeys,'recipient_region'])[NICHE_STATISTICS_METRICS].mean().unstack('recipient_region')
                if all(region in cr.columns.levels[1] for region in ['detected_crypt','outside_detected_crypt']):
                    cr=cr.loc[:,pd.IndexSlice[:,['detected_crypt','outside_detected_crypt']]].dropna()
                    cc=pd.DataFrame({m:cr[(m,'detected_crypt')]-cr[(m,'outside_detected_crypt')] for m in NICHE_STATISTICS_METRICS}).reset_index()
                    store('crypt_paired',cc,['marker','hop',*basekeys],NICHE_STATISTICS_METRICS)
                # Mean absolute single-source sensitivity at a fixed recipient; excludes double-positive precursor edits.
                l=l[((l.marker=='Agr2')&(l.source_state=='precursor_only'))|
                    ((l.marker=='Chroma')&(l.source_state=='precursor_only'))|l.marker.isin(['Lysozyme','Serotonin'])].copy()
                for metric in NICHE_STATISTICS_METRICS: l['abs_'+metric]=l[metric].abs()
                idkeys=['organoid_str','recipient','hop',*basekeys]
                mm=l.groupby([*idkeys,'marker'])[NICHE_STATISTICS_METRICS+['abs_'+m for m in NICHE_STATISTICS_METRICS]].mean().unstack('marker')
                for a,b in itertools.combinations(['Agr2','Lysozyme','Chroma','Serotonin'],2):
                    if a not in mm.columns.levels[1] or b not in mm.columns.levels[1]: continue
                    subset=mm.loc[:,pd.IndexSlice[:,[a,b]]].dropna()
                    if not len(subset): continue
                    record=pd.DataFrame(index=subset.index)
                    for metric in NICHE_STATISTICS_METRICS:
                        av,bv=subset[('abs_'+metric,a)],subset[('abs_'+metric,b)]
                        record['a_'+metric]=subset[(metric,a)]; record['b_'+metric]=subset[(metric,b)]
                        record['strength_a_'+metric]=av; record['strength_b_'+metric]=bv
                        record['dominance_'+metric]=(av-bv)/(av+bv).where(av+bv>1e-10)
                    record=record.reset_index().assign(marker_a=a,marker_b=b)
                    columns=[c for c in record if c.endswith(tuple(NICHE_STATISTICS_METRICS))]
                    store('dominance',record,['marker_a','marker_b','hop',*basekeys],columns)
                    if n in [counts[0],counts[-1]]: dominance_cases.append(record)
                    if seed==config['seeds'][0] and n==counts[0]:
                        support.append(dict(analysis='dominance',marker_a=a,marker_b=b,hop=None,fold=fold,
                            n_recipients=len(record[['organoid_str','recipient','hop']].drop_duplicates()),n_organoids=record.organoid_str.nunique()))
                # Backup presence contrasts on matched within-organoid context strata.
                matched=s.merge(cm,on='case_id',validate='one_to_one')
                mk=[*context_keys,'marker',*basekeys]
                bm=matched.groupby([*mk,'backup_present'])[NICHE_STATISTICS_METRICS].mean().unstack('backup_present').dropna()
                if len(bm):
                    bc=pd.DataFrame({m:bm[(m,1)]-bm[(m,0)] for m in NICHE_STATISTICS_METRICS}).reset_index()
                    for m in NICHE_STATISTICS_METRICS:
                        bc['backup0_'+m]=bm[(m,0)].to_numpy(); bc['backup1_'+m]=bm[(m,1)].to_numpy()
                    store('backup',bc,['marker','hop',*basekeys],NICHE_STATISTICS_METRICS+['backup0_'+m for m in NICHE_STATISTICS_METRICS]+['backup1_'+m for m in NICHE_STATISTICS_METRICS])
                # Average source-pair draws inside recipient before organoid means.
                pmetrics=[c for c in p if c.endswith(('_z','_raw','_relative'))]
                for selection,mask in [('all',np.ones(len(p),dtype=bool)),
                    ('one_each_in_hop',(p.count_lyso==1)&(p.count_sero==1)&(p.cross_positive==0))]:
                    pp=p[mask].groupby(['organoid_str','recipient','hop',*basekeys])[pmetrics].mean().reset_index()
                    store('pairs',pp.assign(selection=selection),['selection','hop',*basekeys],pmetrics)
            print(f'Summarized fold={fold} seed={seed}',flush=True)
    pd.DataFrame(support).to_csv(out/'matched_support.csv',index=False)
    for name,parts in results.items():
        if not parts: continue
        frame=pd.concat(parts,ignore_index=True)
        frame.to_csv(out/f'{name}_organoid.csv.gz',index=False)
        keys=[c for c in frame if c not in ['organoid_str','seed'] and not c.endswith(('_z','_raw','_relative'))]
        metrics=[c for c in frame if c.endswith(('_z','_raw','_relative'))]
        cluster_summary(frame,keys,metrics,draws=draws).to_csv(out/f'{name}_summary.csv',index=False)
        frame.groupby([*keys,'seed'])[metrics].mean().reset_index().to_csv(out/f'{name}_seed_summary.csv',index=False)
        if 'n' in keys and frame.n.max()>0:
            ek=[k for k in keys if k!='n']
            # Endpoints are paired within organoid AND fitted seed before averaging seeds.
            low=frame[frame.n==counts[0]].set_index([*ek,'organoid_str','seed'])[metrics]
            high=frame[frame.n==counts[-1]].set_index([*ek,'organoid_str','seed'])[metrics]
            endpoint=(high-low).dropna(how='all').reset_index()
            cluster_summary(endpoint,ek,metrics,draws=draws).to_csv(out/f'{name}_endpoint.csv',index=False)
            endpoint.groupby([*ek,'seed'])[metrics].mean().reset_index().to_csv(out/f'{name}_endpoint_seeds.csv',index=False)
    dc=pd.concat(dominance_cases,ignore_index=True)
    dc.to_csv(out/'dominance_endpoint_cases.csv.gz',index=False)
    # A switch is an absolute-sensitivity rank change, not an additive share of the prediction.
    ix=['organoid_str','recipient','hop','marker_a','marker_b','seed']
    wide=dc.pivot(index=ix,columns='n',values='dominance_delta_z').dropna()
    switches=wide.reset_index()[ix].copy()
    switches['a_to_b']=((wide[counts[0]]>0)&(wide[counts[-1]]<0)).to_numpy().astype(float)
    switches['b_to_a']=((wide[counts[0]]<0)&(wide[counts[-1]]>0)).to_numpy().astype(float)
    switches.to_csv(out/'dominance_switch_cases.csv.gz',index=False)
    cluster_summary(switches,['hop','marker_a','marker_b'],['a_to_b','b_to_a'],draws=draws).to_csv(out/'dominance_switch_summary.csv',index=False)
    # Matching balance and support at the static case level; no model outcomes used for matching.
    balance=sm.merge(cm,on='case_id')
    balance.groupby(['marker','hop','backup_present']).agg(n_cases=('case_id','size'),n_organoids=('organoid_str','nunique'),
        mean_n=('observed_n','mean'),mean_degree=('recipient_degree','mean'),mean_lgr5_fraction=('local_lgr5_fraction','mean')).to_csv(out/'backup_balance.csv')
    covariates=['observed_n','recipient_degree','local_lgr5_fraction']
    weighted=balance.groupby([*context_keys,'marker','backup_present'])[covariates].mean().reset_index()
    weighted=weighted.groupby(['organoid_str','marker','hop','backup_present'])[covariates].mean().reset_index()
    weighted.groupby(['marker','hop','backup_present'])[covariates].mean().to_csv(out/'backup_balance_analysis_weighted.csv')
    dominance=pd.concat(results['dominance'],ignore_index=True)
    dm=['dominance_delta_z','dominance_delta_relative']
    observed=dominance[dominance.n==0].merge(orgs,on='organoid_str',validate='many_to_one')
    for stratum in ['timepoint','n_bin']:
        cluster_summary(observed,['marker_a','marker_b','hop',stratum],dm,draws=draws).to_csv(out/f'dominance_observed_{stratum}.csv',index=False)
    # Local sensitivity: nearest bracketing grid points to each graph's actual N.
    # This avoids making the full endpoint extrapolation the sole size test.
    local=[]
    actual=orgs.set_index('organoid_str').observed_n
    for keys,g in dominance[dominance.n>0].groupby(['organoid_str','marker_a','marker_b','hop','seed']):
        observed_n=actual.loc[keys[0]]
        if not counts[0] <= observed_n <= counts[-1]: continue
        g=g.sort_values('n').reset_index(drop=True)
        hi=min(max(int(np.searchsorted(g.n.to_numpy(),observed_n)),1),len(g)-1); lo=hi-1
        diff=(g.loc[hi,dm]-g.loc[lo,dm])/(np.log(g.loc[hi,'n'])-np.log(g.loc[lo,'n']))
        local.append(dict(zip(['organoid_str','marker_a','marker_b','hop','seed'],keys))|diff.to_dict())
    local=pd.DataFrame(local)
    local.to_csv(out/'dominance_local_slopes.csv',index=False)
    cluster_summary(local,['marker_a','marker_b','hop'],dm,draws=draws).to_csv(out/'dominance_local_slope_summary.csv',index=False)
    return out

## Film figure definitions

These study-specific definitions build the summaries or panels displayed below. Cohort selection, marker labels, thresholds and figure layouts remain visible here; shared numerical and plotting primitives are imported from `src/`. Defining these functions does not run inference or load results.

In [ ]:
from matplotlib.lines import Line2D

FILM_ROUTE_LABELS = {
    'full': 'FiLM + head vary',  # Figure label for full.
    'film_only': 'FiLM varies; head fixed',  # Figure label for film_only.
    'head_only': 'Head varies; FiLM fixed',  # Figure label for head_only.
    'layer1_only': 'Only FiLM layer 1 varies',  # Figure label for layer1_only.
    'layer2_only': 'Only FiLM layer 2 varies',  # Figure label for layer2_only.
}  # Display labels for the size-conditioning routes.
FILM_COLORS = dict(zip(FILM_ROUTE_LABELS, ['#222222','#d45a31','#2878b5','#29945b','#9467bd']))  # Display colors for the size-conditioning routes.

def film_axis_style(ax, counts):
    ax.set_xscale('log')
    ax.set_xticks([counts[0],counts[len(counts)//2],counts[-1]],
                  labels=[str(counts[0]),str(counts[len(counts)//2]),str(counts[-1])])
    ax.minorticks_off()
    ax.tick_params(labelsize=8)
    ax.spines[['top','right']].set_visible(False)

def film_route_curves(table, markers, counts, *, metric='delta_z', routes=None, minimum=10):
    routes = routes or ['full','film_only','head_only']
    labels={'delta_z':'Δ transformed curvature','delta_fixed_reference':'ΔK / Kref(361)',
            'delta_relative':'ΔK / Kref(N)'}
    fig,axes=plt.subplots(2,len(markers),squeeze=False,figsize=(3.1*len(markers),6.8),sharex=True)
    for row,hop in enumerate([1,2]):
        for col,marker in enumerate(markers):
            ax=axes[row,col];ax.axhline(0,color='0.7',lw=.7)
            for route in routes:
                group=table[(table.metric==metric)&(table.source_marker_name==marker)&(table.hop==hop)&(table.route==route)].sort_values('n')
                valid=group.n_organoids>=minimum
                ax.plot(group.n,group['mean'].where(valid),color=FILM_COLORS[route],label=FILM_ROUTE_LABELS[route],lw=1.5)
                ax.fill_between(group.n,group.ci_low.where(valid),group.ci_high.where(valid),color=FILM_COLORS[route],alpha=.1)
            film_axis_style(ax,counts)
            if row==0:ax.set_title(f'Ablate {marker}',fontsize=10)
            if col==0:ax.set_ylabel(f'Hop {hop}\n{labels.get(metric,metric)}')
            if row==1:ax.set_xlabel('Supplied N')
    handles=[Line2D([],[],color=FILM_COLORS[r],label=FILM_ROUTE_LABELS[r]) for r in routes]
    fig.legend(handles=handles,loc='upper center',bbox_to_anchor=(.5,.955),ncol=len(routes),fontsize=9)
    fig.suptitle('LGR5-positive centers: separating FiLM and head size inputs',y=.998)
    fig.tight_layout(rect=(0,0,1,.9))
    return fig

def film_scaling_heatmap(table, counts, *, hop=1, metric='delta_relative'):
    selected=table[(table.hop==hop)&(table.metric==metric)&((table.marker_a=='LGR5')|(table.marker_b=='LGR5'))].copy()
    selected['other']=np.where(selected.marker_a=='LGR5',selected.marker_b,selected.marker_a)
    markers=sorted(selected.other.unique())
    if not markers:raise ValueError('No supported common-center marker comparisons.')
    fig,axes=plt.subplots(1,3,figsize=(16, max(3.8,len(markers)*.55)),squeeze=False)
    for ax,route in zip(axes[0],['full','film_only','head_only']):
        group=selected[selected.route==route]
        matrix=group.pivot(index='other',columns='n',values='mismatch').reindex(index=markers,columns=counts)
        im=ax.imshow(matrix,aspect='auto',cmap='viridis',vmin=0,vmax=1)
        ax.set_xticks(range(len(counts)),counts,rotation=45)
        support=group.groupby('other').n_organoids.first()
        ax.set_yticks(range(len(markers)),[f'{m} vs LGR5 (n={support.get(m,0)})' for m in markers],fontsize=9)
        ax.set_title(FILM_ROUTE_LABELS[route],fontsize=11)
        ax.set_xlabel('Supplied N')
    fig.suptitle(f'Beyond common positive scaling: matched centers, hop {hop}, {metric}\nReference N=361; each row uses its own common-support cohort',y=1.02)
    fig.subplots_adjust(left=.12,right=.87,bottom=.22,top=.82,wspace=.75)
    color_axis=fig.add_axes([.92,.22,.012,.60])
    fig.colorbar(im,cax=color_axis,label='Unexplained response norm fraction')
    return fig

def film_center_interaction_curves(table,seeds,markers,counts,*,metric='interaction_z',minimum=10):
    fig,axes=plt.subplots(2,len(markers),squeeze=False,figsize=(3.1*len(markers),6.5),sharex=True)
    for row,hop in enumerate([1,2]):
        for col,marker in enumerate(markers):
            ax=axes[row,col];ax.axhline(0,color='0.6',lw=.7)
            group=table[(table.kind=='center_neighbor')&(table.marker_b==marker)&(table.hop==hop)&(table.metric==metric)].sort_values('n')
            valid=group.n_organoids>=minimum
            for seed,sg in seeds[(seeds.kind=='center_neighbor')&(seeds.marker_b==marker)&(seeds.hop==hop)].groupby('seed'):
                sg=sg.sort_values('n')
                if valid.any():ax.plot(sg.n,sg[metric],color='0.6',lw=.8,alpha=.65)
            ax.plot(group.n,group['mean'].where(valid),color='#9467bd',lw=1.7)
            ax.fill_between(group.n,group.ci_low.where(valid),group.ci_high.where(valid),color='#9467bd',alpha=.18)
            film_axis_style(ax,counts)
            if row==0:ax.set_title(f'Center LGR5 × {marker}',fontsize=10)
            if col==0:ax.set_ylabel(f'Hop {hop}\n'+('Interaction / Kref(N)' if metric=='interaction_relative' else 'Interaction in transformed curvature'))
            if row==1:ax.set_xlabel('Supplied N')
    fig.suptitle('Factorial center–neighbor interaction: both − center-only − neighbor-only + intact\nPurple: organoid mean and conditional 95% CI; gray: individual model seeds',y=.995)
    fig.tight_layout(rect=(0,0,1,.9))
    return fig

def film_neighbor_interaction_heatmap(endpoints,markers,*,minimum=10):
    fig,axes=plt.subplots(2,2,figsize=(13,11))
    for row,metric in enumerate(['interaction_z','interaction_relative']):
        eligible=endpoints[(endpoints.kind=='neighbor_neighbor')&(endpoints.metric==metric)&(endpoints.n_organoids>=minimum)]
        vmax=max(1e-8,float(eligible['mean'].abs().max()))
        for col,hop in enumerate([1,2]):
            matrix=np.full((len(markers),len(markers)),np.nan)
            support=np.zeros_like(matrix)
            for item in eligible[eligible.hop==hop].itertuples():
                i,j=markers.index(item.marker_a),markers.index(item.marker_b)
                matrix[i,j]=matrix[j,i]=item.mean;support[i,j]=support[j,i]=item.n_organoids
            ax=axes[row,col];im=ax.imshow(matrix,cmap='RdBu_r',vmin=-vmax,vmax=vmax)
            ax.set_xticks(range(len(markers)),markers,rotation=45,ha='right')
            ax.set_yticks(range(len(markers)),markers)
            for i,j in zip(*np.where(np.isfinite(matrix))):ax.text(j,i,f'{matrix[i,j]:.2g}\nn={int(support[i,j])}',ha='center',va='center',fontsize=7)
            ax.set_title(f'Hop {hop}: {metric}')
            fig.colorbar(im,ax=ax,shrink=.75)
    fig.suptitle('Change in neighbor–neighbor interaction: N=791 minus N=165\nDistinct source cells; LGR5-positive center retained; blank = insufficient support',y=.995)
    fig.tight_layout(rect=(0,0,1,.93))
    return fig

def film_hidden_interaction_heatmap(table,counts,*,kind='center_neighbor',minimum=10):
    selected=table[(table.kind==kind)&(table.metric=='h2_interaction_ratio')&(table.n_organoids>=minimum)].copy()
    selected['pair']=selected.marker_a+' × '+selected.marker_b
    pairs=sorted(selected.pair.unique())
    fig,axes=plt.subplots(1,2,figsize=(13,max(4,len(pairs)*.35)))
    vmax=max(.05,float(selected['mean'].max()))
    for ax,hop in zip(axes,[1,2]):
        matrix=selected[selected.hop==hop].pivot(index='pair',columns='n',values='mean').reindex(index=pairs,columns=counts)
        im=ax.imshow(matrix,aspect='auto',cmap='viridis',vmin=0,vmax=vmax)
        ax.set_xticks(range(len(counts)),counts,rotation=45);ax.set_yticks(range(len(pairs)),pairs,fontsize=8)
        ax.set_title(f'Layer 2, hop {hop}');ax.set_xlabel('Supplied N')
    fig.suptitle(f'Non-additivity before the prediction head: {kind}',y=1.01)
    fig.subplots_adjust(left=.2,right=.85,bottom=.17,top=.9,wspace=.85)
    color_axis=fig.add_axes([.92,.17,.012,.73])
    fig.colorbar(im,cax=color_axis,label='Relative hidden non-additivity')
    return fig

def film_neighbor_interaction_size_heatmap(table,counts,*,minimum=10):
    selected=table[(table.kind=='neighbor_neighbor')&(table.n_organoids>=minimum)].copy()
    selected['pair']=selected.marker_a+' × '+selected.marker_b
    pairs=sorted(selected.pair.unique())
    fig,axes=plt.subplots(2,2,figsize=(16,max(10,len(pairs)*.65)),layout='constrained')
    for row,metric in enumerate(['interaction_z','interaction_relative']):
        group=selected[selected.metric==metric]
        vmax=max(1e-8,float(group['mean'].abs().max()))
        for col,hop in enumerate([1,2]):
            panel=group[group.hop==hop]
            matrix=panel.pivot(index='pair',columns='n',values='mean').reindex(index=pairs,columns=counts)
            ax=axes[row,col];im=ax.imshow(matrix,aspect='auto',cmap='RdBu_r',vmin=-vmax,vmax=vmax)
            ax.set_xticks(range(len(counts)),counts,rotation=45)
            ax.set_yticks(range(len(pairs)),pairs,fontsize=8)
            ax.set_xlabel('Supplied N');ax.set_title(f'Hop {hop}: {metric}')
            fig.colorbar(im,ax=ax,shrink=.7)
    fig.suptitle('Neighbor–neighbor factorial interactions across size\nOriginally LGR5-positive centers; distinct source cells; blank = insufficient support')
    return fig

## Niche hypotheses: intact recipients, source states, and redundancy

This section uses the saved **FiLM model without geometry in the head**. All 15 checkpoints have already been evaluated; the default below only reads cached results and displays figures. It can be run independently of the training cells above.

**Biological constraints:** recipients retain their full original fate vector. Editing a source clears one marker bit, not the cell or its other markers. Serotonin-cell apical constriction is a qualitative observation supplied by the investigator; neither cell apical constriction nor mechanical forces are measured here. Crypt labels describe detectable, sufficiently mature structures in destructive snapshots. “No crypt detected” does not establish absence of immature crypts, and N is not a longitudinal clock.

Sources are sampled within three distinct states for each proposed precursor/successor family: precursor-marker only, double positive, and mature-marker only. These are measured marker states, not tracked lineage transitions. At most three source cells per state per organoid are selected with a fixed seed. Full graphs preserve the two-layer receptive field of all recipients at exact hop 1 or 2.

**Effect convention:** `ΔK = predicted K after source-marker removal − predicted K intact`. A negative value means removal lowers predicted curvature at the unchanged recipient. The plotted normalized effect is `ΔK × Â_train-fold(N)/(4π)`, using the already-saved geometric calibration. For observed-size results, N is the actual cell count; for sweeps, graph, fate vectors and crypt labels stay fixed while the FiLM and head size inputs both vary.

**Dominance** compares the mean absolute single-source effects of two candidate marker groups on the *same* LGR5-positive recipient at the same hop: `D = (strength A − strength B)/(strength A + strength B)`. Positive D favors A; negative D favors B. It is a relative per-cell sensitivity measure, not an additive fraction of the prediction. Agr2 and Chroma in these comparisons use only precursor-only sources; Lysozyme and Serotonin include their double-positive and mature-only states.

Intervals are 1,000 organoid-bootstrap intervals conditional on the fitted models and geometric calibration; seeds are averaged inside each organoid. Endpoint differences are paired. All seed summaries, support tables, and raw case effects are saved alongside the report.

In [ ]:
from IPython.display import display, Markdown


# Paths and outputs

print("Cached analysis:", NICHE_OUTPUT_DIR)
display(pd.Series(json.loads((NICHE_OUTPUT_DIR / "validation.json").read_text()), name="Validation"))
# Study-specific functions are defined in this notebook.
niche_figures = niche_save_figures(NICHE_OUTPUT_DIR)


### Recipient specificity

Both recipient classes are present around the same edited source in these paired comparisons. Thus differences cannot be attributed to selecting a different Lysozyme/Serotonin source cohort. LGR5-negative is a heterogeneous comparator. A recipient difference describes context-dependent model processing; it does not isolate the causal effect of LGR5 expression.

In [ ]:
display(niche_figures['01_recipient_fields'])
plt.close(niche_figures['01_recipient_fields'])
display(niche_figures['02_recipient_specificity'])
plt.close(niche_figures['02_recipient_specificity'])

### Source states and changes in relative sensitivity

The source-state curves use different eligible cohorts and are descriptive. The dominance and signed-effect comparisons use matched recipients containing both candidate source groups and keep this cohort fixed across N. Compare the normalized and transformed-target dominance curves to assess whether the result is driven by the output scale.

In [ ]:
display(niche_figures['03_source_states'])
plt.close(niche_figures['03_source_states'])
display(niche_figures['04_matched_dominance'])
plt.close(niche_figures['04_matched_dominance'])
display(niche_figures['05_matched_signed_effects'])
plt.close(niche_figures['05_matched_signed_effects'])

### Lysozyme–Serotonin redundancy

Backup-context comparisons match the same source cell, organoid, exact hop, recipient marker signature and detected-crypt region, plus coarsened recipient degree and local LGR5 fraction. The backup is a distinct cell within two hops of the recipient; sources coexpressing the backup marker are excluded. Residual neighborhood differences remain.

The factorial experiment selects distinct Lysozyme-positive and Serotonin-positive sources in the same hop of an intact LGR5 recipient. The interaction is `both removed − Lysozyme removed − Serotonin removed + intact`. With negative single-removal effects, a negative interaction is consistent with buffering/redundancy in the model. “Exactly one of each” refers to the tested hop, not the entire receptive field; other marker bits and other-hop sources remain.

In [ ]:
display(niche_figures['06_redundancy'])
plt.close(niche_figures['06_redundancy'])
display(niche_figures['15_transformed_redundancy_checks'])
plt.close(niche_figures['15_transformed_redundancy_checks'])

### Detected crypt context

Detected-crypt labels remain at their observed values during sweeps. Curves for different regions do not represent a crypt transitioning between states. The paired plot compares LGR5 recipients inside and outside a detected crypt around the same edited source; the supporting cohort can be small.

In [ ]:
display(niche_figures['07_crypt_context'])
plt.close(niche_figures['07_crypt_context'])
display(niche_figures['13_paired_crypt_context'])
plt.close(niche_figures['13_paired_crypt_context'])

### Observed size, local size sweeps, and acquisition timepoints

Observed-size bins contain different organoids. Local derivatives use only the adjacent supplied-N grid points bracketing each organoid's actual N, and exclude organoids outside 165–791. The two day-4.5 acquisition groups remain separate. Agreement with a fixed-context sweep strengthens a size-associated model finding but does not identify longitudinal development.

In [ ]:
display(niche_figures['08_observed_dominance'])
plt.close(niche_figures['08_observed_dominance'])
display(niche_figures['11_local_size_sensitivity'])
plt.close(niche_figures['11_local_size_sensitivity'])
display(niche_figures['12_snapshot_timepoints'])
plt.close(niche_figures['12_snapshot_timepoints'])

### Measured marker-state prevalence and curvature

Prevalence uses every cell in the saved cohort, not the capped source sample. Fractions are averaged per organoid. Measured Gaussian curvature is the raw exported cell-patch target, in dataset units; it is not cell apical area or a measurement of constriction. Marker groups can overlap. The supplementary recipient-phenotype curves have different eligible source cohorts and are descriptive.

In [ ]:
display(niche_figures['09_state_prevalence'])
plt.close(niche_figures['09_state_prevalence'])
display(niche_figures['10_measured_curvature'])
plt.close(niche_figures['10_measured_curvature'])
display(niche_figures['14_other_recipient_phenotypes'])
plt.close(niche_figures['14_other_recipient_phenotypes'])

### Findings report

The report below is generated from the completed cached analyses. Its supporting tables and standalone PNG/PDF figures are in `NICHE_OUTPUT_DIR`.

In [ ]:
display(Markdown((NICHE_OUTPUT_DIR / "REPORT.md").read_text()))

## Optional: FiLM route and factorial diagnostics

The following diagnostic applies specifically to FiLM architectures: independently vary conditioning and head size inputs on fixed recipients, inspect common scaling and explicit factorial contrasts. This is an internal network intervention; it belongs with ablation, not general-purpose clustering.

<a id="lgr5-film-diagnostics"></a>

### FiLM interactions around LGR5-positive centers

This section contains the completed checkpoint-based diagnostics for the **FiLM model
without geometry in the head**. It can be executed independently, starting with the imports
below. By default it reads completed diagnostics and regenerates figures. Set
`LGR5_RUN_INFERENCE=True` only to run inference again; checkpoint-level completion files
allow resuming. No model or baseline is retrained. `LGR5_RECOMPUTE_SUMMARIES=True` recomputes
summaries from saved diagnostic cases without inference.

All 767 originally LGR5-positive centers in the saved size-sweep manifests are retained:
461 held-out organoids, five folds, three seeds, and N = 165, 201, 244, 297, 361, 439, 535,
650, 791. The reference is **N=361**. Center coexpression is retained and reported below.
All source cells and neighborhoods stay fixed across N, seeds, and diagnostic routes.

We distinguish five routes: both FiLM and head vary; FiLM varies with head fixed; head
varies with FiLM fixed; only FiLM layer 1 varies; only FiLM layer 2 varies. All other size
inputs stay at N=361. These are internal network interventions, not simulated organoids.
Full-route predictions are checked against the saved original sweeps for every case/N/seed.

Geometric normalization uses the existing shared training-fold references. The route plots
also show transformed curvature and curvature divided by **Kref(361)**, avoiding size trends
introduced just by changing the denominator. The GNN/head split concerns the explicit size
input: the fixed head remains nonlinear in the changing GNN representation.

All intervals are **pointwise organoid-bootstrap 95% intervals conditional on fitted models
and reference curves**. Seeds are averaged within organoids, and seed-specific results are
exported separately. Endpoint contrasts are paired within case, organoid, and seed. These
are exploratory analyses without multiplicity-adjusted significance claims.

In [ ]:
if SHOW_FILM_ROUTES:

    from src.analysis.conditioning.film import run_diagnostics
    from src.analysis.conditioning.response_statistics import summarize_diagnostics

    # Study-specific functions are defined in this notebook.

    # Paths and outputs
    LGR5_RUN_DIR = PROJECT_ROOT / 'results_experiments/size_conditioned_ablation' / LGR5_RUN_NAME  # Training run to create or saved run to analyze.
    LGR5_OUTPUT_DIR = LGR5_RUN_DIR / 'lgr5_film_diagnostics'  # Directory for this analysis’s tables and figures.

    # Inference resources

    # Statistical summaries
    LGR5_CONFIG = json.loads((LGR5_OUTPUT_DIR / 'settings.json').read_text())  # Settings for the observed-cell or intervention study.
    LGR5_COUNTS = LGR5_CONFIG['counts']  # Supplied count grid recorded in the selected diagnostic.
    LGR5_MARKERS = LGR5_CONFIG['marker_names']  # Marker order recorded by the selected training run.
    lgr5_tables = {name: pd.read_csv(LGR5_OUTPUT_DIR / f'{name}.csv') for name in [
        'sampling_support', 'reproduction_checks', 'center_coexpression_support',
        'effect_summary', 'effect_seed_summary', 'effect_endpoint_summary', 'effect_endpoint_seeds',
        'interaction_summary', 'interaction_seed_summary', 'interaction_endpoint_summary', 'interaction_endpoint_seeds',
        'common_scaling', 'common_scaling_seeds', 'common_scaling_support', 'route_interaction_summary',
    ]}
    LGR5_FIGURE_DIR = LGR5_OUTPUT_DIR / 'figures'  # Destination for the generated figures.
    LGR5_FIGURE_DIR.mkdir(exist_ok=True)
    def lgr5_show_save(fig, name):
        for extension in ['png', 'pdf']:
            fig.savefig(LGR5_FIGURE_DIR / f'{name}.{extension}', dpi=150, bbox_inches='tight')
        display(fig)
        plt.close(fig)
    print('Source run:', LGR5_RUN_NAME)
    display(lgr5_tables['sampling_support'], lgr5_tables['center_coexpression_support'])
    print('Completed checkpoints:', len(lgr5_tables['reproduction_checks']))
    print('Maximum difference from original saved ΔK:', lgr5_tables['reproduction_checks'].max_saved_raw_difference.max())


#### 1. Changes in marker preference beyond common scaling

For each pair of neighboring markers and hop, retain only centers where both selected
source-positive cells exist. This common-support cohort is fixed across N and seeds.
Pairs may involve two markers on the same source cell in this sensitivity comparison;
the neighbor–neighbor factorial experiment below instead requires distinct cells.
A joint seven-marker cohort would be restrictive; the matched-pair analysis makes support
explicit and avoids averaging different marker-specific cohorts as though they were matched.

Fit the best nonnegative multiplier $a(N)$ of the two-marker mean response at N=361.
The heatmap shows $\|e(N)-a(N)e(361)\|/\|e(N)\|$: 0 means a common positive scale fits;
1 means it explains none of the response vector's norm. This is a **norm fraction, not a
fraction of variance**. Pair-specific marker residuals, gains, bootstrap intervals, and
seed results are exported. Near-zero vectors are undefined. The bootstrap refits the
multiplier using whole paired organoid curves; positive mismatch intervals alone are not
a calibrated test against zero. Both geometric and transformed scales are shown.

In [ ]:
if SHOW_FILM_ROUTES:
    for lgr5_metric in ['delta_relative', 'delta_z']:
        for lgr5_hop in [1, 2]:
            lgr5_show_save(film_scaling_heatmap(lgr5_tables['common_scaling'], LGR5_COUNTS,
                hop=lgr5_hop, metric=lgr5_metric), f'common_scaling_{lgr5_metric}_hop{lgr5_hop}')
    display(lgr5_tables['common_scaling'].query(
        "marker_a == 'LGR5' and marker_b == 'Lysozyme' and hop == 1 and metric == 'delta_z' and n == 165"))
    display(lgr5_tables['common_scaling_seeds'].query(
        "marker_a == 'LGR5' and marker_b == 'Lysozyme' and hop == 1 and metric == 'delta_z' and n == 165"))


#### 2. Separate GNN conditioning from the explicit head size input

The first two figures use transformed curvature and a **fixed** geometric denominator
Kref(361). A varying Kref(N) is deliberately avoided for route attribution. The third figure
isolates the two FiLM layers. These routes need not add up to the full response: the
network is nonlinear. The exported `route_interaction_summary.csv` also measures
`full − FiLM-only − head-only + reference` in transformed curvature at each N.

Each marker panel uses its original source-positive cohort, held fixed across all routes
and sizes. Panels with fewer than ten organoids are masked. The common-scaling analysis
above provides the stricter matched-center comparisons between different markers.

In [ ]:
if SHOW_FILM_ROUTES:
    for lgr5_metric in ['delta_z', 'delta_fixed_reference']:
        lgr5_show_save(film_route_curves(lgr5_tables['effect_summary'], LGR5_MARKERS, LGR5_COUNTS,
            metric=lgr5_metric, minimum=LGR5_MIN_ORGANOIDS), f'route_comparison_{lgr5_metric}')
    lgr5_show_save(film_route_curves(lgr5_tables['effect_summary'], ['LGR5','Lysozyme','AldoB'], LGR5_COUNTS,
        routes=['full','film_only','head_only','layer1_only','layer2_only'], minimum=LGR5_MIN_ORGANOIDS),
        'layer_specific_size_routes')
    display(lgr5_tables['effect_endpoint_summary'].query(
        "metric == 'delta_z' and hop == 1 and source_marker_name in ['LGR5','Lysozyme','AldoB']"))
    display(lgr5_tables['effect_endpoint_seeds'].query(
        "hop == 1 and source_marker_name in ['LGR5','Lysozyme']"))


#### 4. Factorial marker interactions

For the same neighborhood and N, calculate

$$I=f_{-A,-B}-f_{-A}-f_{-B}+f_{\rm intact}.$$

Zero means additive removals on the chosen output scale; positive I means the joint
removal response is more positive than the sum of the single-removal responses.
Center–neighbor tests temporarily remove the center's LGR5 signal and one selected
neighbor marker. All centers were LGR5-positive originally. Neighbor–neighbor tests
keep the center unchanged and remove different markers on **distinct source cells in the
same hop**. The fixed manifest gives 4,745 center–neighbor tests and 5,204 neighbor-pair
tests per N/checkpoint coverage across folds. The 658 candidate pairs using the same
source cell are excluded. These are marker edits, not cell removal or fate conversion.

Transformed and normalized-curvature contrasts are both shown because the nonlinear
inverse target transform can create output non-additivity. The hidden contrast provides
an additional check before the head: its norm divided by the sum of the two single-edit
hidden norms. This unsigned ratio measures non-additivity, not whether curvature shifts
positively or negatively. No individual hidden channels are assigned cell-type meanings.

In [ ]:
if SHOW_FILM_ROUTES:
    for lgr5_metric in ['interaction_z','interaction_relative']:
        lgr5_show_save(film_center_interaction_curves(lgr5_tables['interaction_summary'],
            lgr5_tables['interaction_seed_summary'], LGR5_MARKERS, LGR5_COUNTS,
            metric=lgr5_metric, minimum=LGR5_MIN_ORGANOIDS), f'center_neighbor_{lgr5_metric}')
    lgr5_show_save(film_neighbor_interaction_heatmap(lgr5_tables['interaction_endpoint_summary'],
        LGR5_MARKERS, minimum=LGR5_MIN_ORGANOIDS), 'neighbor_neighbor_endpoint_changes')
    # Study-specific functions are defined in this notebook.
    lgr5_show_save(film_neighbor_interaction_size_heatmap(lgr5_tables['interaction_summary'], LGR5_COUNTS,
        minimum=LGR5_MIN_ORGANOIDS), 'neighbor_neighbor_interactions_across_N')
    for lgr5_kind in ['center_neighbor','neighbor_neighbor']:
        lgr5_show_save(film_hidden_interaction_heatmap(lgr5_tables['interaction_summary'],LGR5_COUNTS,
            kind=lgr5_kind,minimum=LGR5_MIN_ORGANOIDS), f'hidden_nonadditivity_{lgr5_kind}')
    display(lgr5_tables['interaction_endpoint_summary'].query(
        "metric == 'interaction_z' and kind == 'center_neighbor'"))
    display(lgr5_tables['interaction_endpoint_seeds'].query(
        "hop == 1 and ((kind == 'center_neighbor' and marker_b in ['LGR5','Agr2','Chroma','Lysozyme']) or (kind == 'neighbor_neighbor' and marker_a == 'Agr2' and marker_b == 'KI67'))"))


#### Brief report

The report below describes this saved run. Figures, pointwise and paired-endpoint summaries,
seed summaries, common-support counts, inference manifests, and per-case diagnostic tables
are saved under the run's `lgr5_film_diagnostics` directory.

In [ ]:
if SHOW_FILM_ROUTES:
    display(Markdown((LGR5_OUTPUT_DIR / 'REPORT.md').read_text()))
